In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_ecobici_dev")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsmartdataproy26")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

In [0]:
base = f"abfss://raw@{storageName}.dfs.core.windows.net/datasets"
ruta_origen     = f"{base}/viajes"
ruta_checkpoint = f"{base}/_checkpoints/{catalogo}/viajes_raw"
tabla_destino   = f"{catalogo}.bronze.viajes_raw"

In [0]:
print("Origen :", ruta_origen)
print("Destino:", tabla_destino)

In [0]:
display(dbutils.fs.ls(ruta_origen))

In [0]:
viajes_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.inferColumnTypes", "false")
    .option("cloudFiles.schemaLocation", ruta_checkpoint)
    .option("header", "true")
    .option("encoding", "UTF-8")
    .load(ruta_origen)
)

In [0]:
viajes_bronze = (
    viajes_stream
    .withColumn("_archivo_origen", col("_metadata.file_path"))
    .withColumn("_fecha_ingesta", current_timestamp())
)

In [0]:
consulta = (
    viajes_bronze.writeStream
    .option("checkpointLocation", ruta_checkpoint)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(tabla_destino)
)
consulta.awaitTermination()

In [0]:
bronze = spark.table(tabla_destino)
print("Total de filas:", bronze.count())

In [0]:
display(bronze.groupBy("_archivo_origen").count().orderBy("_archivo_origen"))

In [0]:
display(bronze.limit(10))

In [0]:
# ⚠️ SOLO PARA REINICIAR LA CARGA DE VIAJES DESDE CERO
# Borra la tabla Y el checkpoint

# spark.sql(f"DROP TABLE IF EXISTS {tabla_destino}")
# dbutils.fs.rm(ruta_checkpoint, True)